# 3.7 softmax 回归的简洁实现

本节使用 PyTorch 深度学习框架的高级 API 实现 softmax 回归，可直接复用 3.6 节封装的通用训练流程。框架内置的全连接层、交叉熵损失与优化器大幅简化了重复代码，其中 nn.CrossEntropyLoss 将 softmax 运算与交叉熵损失融合实现，从底层解决了数值上溢、下溢的稳定性问题。

**导入依赖与加载数据集**

沿用 Fashion-MNIST 服装分类数据集，设置批量大小为 256，直接调用 d2l 封装的数据加载接口，获取训练集与测试集的数据迭代器。

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

## 3.7.1 定义与初始化模型
softmax回归的输出层为全连接层，使用nn.Sequential串联两层网络：
1.	nn.Flatten()：将4维图像张量(batch, 1, 28, 28)展平为二维特征张量(batch, 784)。PyTorch不会自动调整输入形状，需显式展平以匹配全连接层的输入维度要求。
2.	nn.Linear(784, 10)：全连接输出层，输入维度784（对应28×28像素），输出维度10（对应10个服装类别）。

定义权重初始化函数，通过net.apply()批量应用到整个网络，将全连接层权重初始化为均值0、标准差0.01的正态分布，与从零实现的初始化规则保持一致。


In [ ]:
# PyTorch不会隐式地调整输入的形状。因此，
# 我们在线性层前定义了展平层（flatten），来调整网络输入的形状
net = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)

net.apply(init_weights);

## 3.7.2 Softmax与交叉熵的数值稳定实现
#### 1. 直接计算softmax的数值风险

softmax公式： $$\hat{y_j}=\frac{\exp (o_{j})}{\sum_{k} \exp (o_{k})}$$
当某类原始输出 $o_k$ 数值很大（比如几十、上百）时，$\exp(o_k)$ 会迅速超出浮点数的表示范围，变成 inf（无穷大），导致分子分母都变成无穷，最终概率出现 inf 或 nan，无法正常计算交叉熵损失与梯度。这就是 上溢（overflow） 问题。


#### 2. 减去行最大值消除上溢


为了消除上溢，标准做法是：对每个样本的所有输出，统一减去该行的最大值 $\max(o_k)$。数学上可以证明，分子分母同乘常数 $\exp(-\max(o_k))$，softmax结果不变： $$ \begin{aligned} \hat{y_j} & =\frac{\exp \left(o_{j}-\max (o_{k})\right) \cdot \exp \left(\max (o_{k})\right)}{\sum_{k} \exp \left(o_{k}-\max (o_{k})\right) \cdot \exp \left(\max (o_{k})\right)} \ & =\frac{\exp \left(o_{j}-\max (o_{k})\right)}{\sum_{k} \exp \left(o_{k}-\max (o_{k})\right)}. \end{aligned} $$
减法之后，最大的指数项变为 $\exp(0)=1$，彻底解决了上溢。 

但其余所有项都变成了负数。如果某个类别输出 $o_j$ 比最大值小很多，$o_j-\max(o_k)$ 就会是绝对值很大的负数，此时 $\exp(o_j-\max(o_k))$ 会极其接近0，受浮点数精度限制直接被舍入为0，这就是 下溢（underflow） 。

后续计算交叉熵时要对概率取对数：$\log(\hat{y}_j)=\log(0)=-\infty$，最终损失变成 -inf，反向传播后梯度全部变成 nan，模型直接失效。

#### 3. Softmax与交叉熵融合计算
既然交叉熵最终要对softmax概率取对数，我们可以把两步运算合并，不单独计算softmax概率，直接从原始输出 $o_j$ 推导交叉熵，让 $\log$ 和 $\exp$ 直接抵消：
$$ \begin{aligned} \log \left(\hat{y_j}\right) & =\log\left(\frac{\exp \left(o_{j}-\max (o_{k})\right)}{\sum_{k} \exp \left(o_{k}-\max (o_{k})\right)}\right) \\ & =o_{j}-\max (o_{k})-\log \left(\sum_{k} \exp \left(o_{k}-\max (o_{k})\right)\right). \end{aligned} $$
这样做的好处：
- 依然保留了「减最大值」的操作，避免上溢；
- 不再单独计算 $\exp(o_j-\max)$ 再取对数，而是直接用原始输出做对数求和，规避了中间概率下溢为0的问题。

这套方法就是 LogSumExp 技巧，是深度学习框架中交叉熵损失的标准实现。



#### 4. PyTorch内置交叉熵损失
nn.CrossEntropyLoss内部集成了softmax运算与交叉熵计算，输入为未归一化的原始输出（logits），无需手动调用softmax，底层采用LogSumExp技巧保证数值稳定性。 设置reduction='none'返回每个样本的独立损失，与3.6手写损失函数的输出格式一致，可直接复用通用训练函数。


In [ ]:
loss = nn.CrossEntropyLoss(reduction='none')

## 3.7.3 定义优化算法
使用小批量随机梯度下降SGD作为优化器，学习率设为0.1，与从零实现的超参数保持一致。通过 net.parameters() 自动获取网络中全部可训练参数。


In [ ]:
trainer = torch.optim.SGD(net.parameters(), lr=0.1)

## 3.7.4 模型训练

直接调用 3.6 节封装的通用训练函数 d2l.train_ch3 ，该函数已通过分支判断兼容 PyTorch 内置优化器，无需修改训练逻辑。设置训练轮数为 10 个迭代周期，训练过程中会实时绘制训练损失、训练精度与测试精度曲线。

In [ ]:
num_epochs = 10
d2l.train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)